# Heat transport in W7-X

TORAX evolves the electron and ion temperatures on the W7-X geometry from GVEC: first with constant diffusivities, where the steady state is known from the heat balance, then with the stellarator neoclassical model, and finally a scan of the heating power.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import plasma_plots  # noqa: F401

from slowcooker import base_config, gvec_geometry, merge, profiles, run, scan
from slowcooker.data import gvec_equilibrium

geometry = gvec_geometry(gvec_equilibrium("w7x"), rho=np.linspace(0.02, 1.0, 40))

## Constant diffusivities

`base_config` writes a TORAX config for the geometry: $\iota$ held fixed (no current diffusion, the right setting for a stellarator with little current), the temperatures evolved, a flat density, a central heating and constant $\chi$. The output of `run` becomes one dataset over `t` and `rho` with `profiles`.

In [ ]:
config = base_config(geometry, density=3e19, heating=2e6, chi=0.5, t_final=1.0, dt=0.05)
data = profiles(run(config))
data.T_e.plasma.plot.profiles(over="t")

In the steady state the heat flowing through each surface equals the power deposited inside it, $P(\rho) = -n\chi V'\langle|\nabla\rho|^2\rangle\, \partial T/\partial\rho$. The stored energy levels off as the profiles settle.

In [ ]:
data.W_thermal_total.plasma.plot.timeseries(logy=False)

## Neoclassical transport

`slowcooker.neoclassical` registers a stellarator neoclassical model with TORAX (`model_name = "stellarator_neoclassical"`): the $1/\nu$ and $\sqrt\nu$ regimes with the effective ripple $\epsilon_\mathrm{eff}$, and the ambipolar radial electric field. It adds to a small turbulent background here. Its prefactors still have to be checked against the literature and a DKES/MONKES database (see the module's documentation), so the numbers are indicative.

In [ ]:
from slowcooker.neoclassical import register

register()
neoclassical = merge(
    config,
    {
        "transport": {
            "core_transport_models": {
                "constant": {"chi_i": 0.1, "chi_e": 0.1},
                "neo": {"model_name": "stellarator_neoclassical", "epsilon_eff": 0.01},
            }
        }
    },
)
final = profiles(run(neoclassical)).isel(t=-1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
final.chi_turb_i.plasma.plot.lineout(x="rho", ax=axes[0], label="ions")
final.chi_turb_e.plasma.plot.lineout(
    x="rho", ax=axes[0], label="electrons", title="$\\chi$ [m$^2$/s]"
)
final.T_i.plasma.plot.lineout(x="rho", ax=axes[1], label="$T_i$")
final.T_e.plasma.plot.lineout(
    x="rho", ax=axes[1], label="$T_e$", title="temperature [keV]"
)
for ax in axes:
    ax.legend()

## A scan of the heating power

`scan` runs the config for each value of one parameter, named by its path, and stacks the results along a new dimension.

In [ ]:
powers = scan(config, "sources.generic_heat.P_total", [1e6, 2e6, 4e6])
powers.T_e.isel(t=-1).plasma.plot.profiles(x="rho", over="P_total")